Разбивка на train и val

In [ ]:
import os
import shutil
import random

random.seed(42)

source_dir = "simpsons_dataset"
train_dir = "simpsons/train"
val_dir = "simpsons/val"
test_dir = "simpsons/test"

for dir_path in [train_dir, val_dir, test_dir]:
    if os.path.exists(dir_path):
        shutil.rmtree(dir_path)
    os.makedirs(dir_path, exist_ok=True)

for class_name in os.listdir(source_dir):
    class_path = os.path.join(source_dir, class_name)
    if not os.path.isdir(class_path):
        continue
    
    images = os.listdir(class_path)
    random.shuffle(images)
    
    total = len(images)
    
    if total < 3:
        print(f"Пропускаем класс {class_name}: только {total} изображений")
        continue
    
    os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(val_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)
    

    if total == 3:
        train_images = [images[0]]
        val_images = [images[1]]
        test_images = [images[2]]
    elif total == 4:
        train_images = images[:2]
        val_images = [images[2]]
        test_images = [images[3]]
    else:
        train_end = int(0.7 * total)
        val_end = int(0.85 * total)
        
        train_images = images[:train_end]
        val_images = images[train_end:val_end]
        test_images = images[val_end:]
    
    assert not (set(train_images) & set(val_images)), "Пересечение train и val!"
    assert not (set(train_images) & set(test_images)), "Пересечение train и test!"
    assert not (set(val_images) & set(test_images)), "Пересечение val и test!"
    
    for img in train_images:
        shutil.copy(os.path.join(class_path, img), os.path.join(train_dir, class_name, img))
    for img in val_images:
        shutil.copy(os.path.join(class_path, img), os.path.join(val_dir, class_name, img))
    for img in test_images:
        shutil.copy(os.path.join(class_path, img), os.path.join(test_dir, class_name, img))

print("Разбиение завершено. Пересечений нет.")

Разбиение завершено. Пересечений нет.


In [ ]:
import os

train_dir = "simpsons/train"
val_dir = "simpsons/val"

train_files = {}
val_files = {}

for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        train_files[class_name] = set(os.listdir(class_path))

for class_name in os.listdir(val_dir):
    class_path = os.path.join(val_dir, class_name)
    if os.path.isdir(class_path):
        val_files[class_name] = set(os.listdir(class_path))

duplicates = {}
all_classes = set(train_files.keys()) | set(val_files.keys())

for class_name in all_classes:
    train_set = train_files.get(class_name, set())
    val_set = val_files.get(class_name, set())
    common = train_set & val_set
    if common:
        duplicates[class_name] = common

if duplicates:
    print("Найдены дубликаты между train и val:")
    for class_name, files in duplicates.items():
        print(f"Класс {class_name}: {files}")
else:
    print("Дубликатов не найдено.")

Дубликатов не найдено.


In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

val_test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = datasets.ImageFolder(root='simpsons/train', transform=train_transform)
val_dataset = datasets.ImageFolder(root='simpsons/val', transform=val_test_transform)
test_dataset = datasets.ImageFolder(root='simpsons/test', transform=val_test_transform)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=6)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=6)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=6)

num_classes = len(train_dataset.classes)
print(f'Классы: {train_dataset.classes}')
print(f'Всего классов: {num_classes}')
print(f'Размер train: {len(train_dataset)}')
print(f'Размер val: {len(val_dataset)}')
print(f'Размер test: {len(test_dataset)}')

Классы: ['abraham_grampa_simpson', 'agnes_skinner', 'apu_nahasapeemapetilon', 'barney_gumble', 'bart_simpson', 'carl_carlson', 'charles_montgomery_burns', 'chief_wiggum', 'cletus_spuckler', 'comic_book_guy', 'disco_stu', 'edna_krabappel', 'fat_tony', 'gil', 'groundskeeper_willie', 'homer_simpson', 'kent_brockman', 'krusty_the_clown', 'lenny_leonard', 'lionel_hutz', 'lisa_simpson', 'maggie_simpson', 'marge_simpson', 'martin_prince', 'mayor_quimby', 'milhouse_van_houten', 'miss_hoover', 'moe_szyslak', 'ned_flanders', 'nelson_muntz', 'otto_mann', 'patty_bouvier', 'principal_skinner', 'professor_john_frink', 'rainier_wolfcastle', 'ralph_wiggum', 'selma_bouvier', 'sideshow_bob', 'sideshow_mel', 'snake_jailbird', 'troy_mcclure', 'waylon_smithers']
Всего классов: 42
Размер train: 14632
Размер val: 3141
Размер test: 3160


In [9]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.fc1 = nn.Linear(32 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')

model = SimpleCNN(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

num_epochs = 20
best_val_acc = 0.0
best_model_path = 'best_model.pth'

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    val_acc = correct / total
    print(f'Эпоха {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}, Val Acc: {val_acc:.4f}')
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f'  -> Новая лучшая модель! Val Acc: {val_acc:.4f}')

print(f'Обучение завершено. Лучшая точность на val: {best_val_acc:.4f}')

Устройство: cpu
Эпоха 1/20, Loss: 2.4252, Val Acc: 0.4559
  -> Новая лучшая модель! Val Acc: 0.4559
Эпоха 2/20, Loss: 1.8037, Val Acc: 0.5400
  -> Новая лучшая модель! Val Acc: 0.5400
Эпоха 3/20, Loss: 1.5328, Val Acc: 0.5969
  -> Новая лучшая модель! Val Acc: 0.5969
Эпоха 4/20, Loss: 1.3364, Val Acc: 0.6313
  -> Новая лучшая модель! Val Acc: 0.6313
Эпоха 5/20, Loss: 1.1729, Val Acc: 0.6256
Эпоха 6/20, Loss: 1.0634, Val Acc: 0.6673
  -> Новая лучшая модель! Val Acc: 0.6673
Эпоха 7/20, Loss: 0.9645, Val Acc: 0.6909
  -> Новая лучшая модель! Val Acc: 0.6909
Эпоха 8/20, Loss: 0.8670, Val Acc: 0.7160
  -> Новая лучшая модель! Val Acc: 0.7160
Эпоха 9/20, Loss: 0.7921, Val Acc: 0.7030
Эпоха 10/20, Loss: 0.7221, Val Acc: 0.7081
Эпоха 11/20, Loss: 0.6644, Val Acc: 0.7332
  -> Новая лучшая модель! Val Acc: 0.7332
Эпоха 12/20, Loss: 0.6182, Val Acc: 0.7348
  -> Новая лучшая модель! Val Acc: 0.7348
Эпоха 13/20, Loss: 0.5588, Val Acc: 0.7370
  -> Новая лучшая модель! Val Acc: 0.7370
Эпоха 14/20, L

Визуальный тест

In [9]:
print(f"train_loader workers: {train_loader.num_workers}")

train_loader workers: 6


F1 Score на test

In [ ]:
from sklearn.metrics import f1_score, classification_report
import numpy as np


best_model_path = "best_model.pth"
model.load_state_dict(torch.load(best_model_path))
model.to(device)
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
print(f"Accuracy на тесте: {accuracy:.4f}")

f1_macro = f1_score(all_labels, all_preds, average='macro')
print(f"F1-macro на тесте: {f1_macro:.4f}")

print("\nДетальный отчёт по классам:")
print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))

Accuracy на тесте: 0.7725
F1-macro на тесте: 0.5807

Детальный отчёт по классам:
                          precision    recall  f1-score   support

  abraham_grampa_simpson       0.79      0.88      0.83       137
           agnes_skinner       0.50      0.29      0.36         7
  apu_nahasapeemapetilon       0.91      0.79      0.85        94
           barney_gumble       0.52      0.69      0.59        16
            bart_simpson       0.70      0.65      0.67       202
            carl_carlson       0.71      0.33      0.45        15
charles_montgomery_burns       0.61      0.75      0.68       179
            chief_wiggum       0.78      0.82      0.80       148
         cletus_spuckler       0.40      0.50      0.44         8
          comic_book_guy       0.81      0.61      0.69        71
               disco_stu       0.00      0.00      0.00         2
          edna_krabappel       0.78      0.74      0.76        69
                fat_tony       1.00      0.40      0.57     

c:\Users\Yuki\Desktop\University\2_course\Programming\ai\classification_simpsons\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yuki\Desktop\University\2_course\Programming\ai\classification_simpsons\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yuki\Desktop\University\2_course\Programming\ai\classification_simpsons\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels wi